#**Royal Plants SmartTrace — CamemBERT NLP training**

**Goal:** Train two French NLP models (intent classifier and BIO NER), evaluate them, and export compatible model folders to use from FastAPI.

**Data ethics:** No original company Excel is uploaded. The starting examples are synthetic. They use fake identifiers and source-derived field types; not authentic employee instructions. Synthetic test scores are not a production benchmark.

In [ ]:
%pip install \
    "huggingface-hub==0.36.2" \
    "diffusers==0.35.2" \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "datasets==4.8.5"

In [ ]:
%pip install jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.2 MB/s  0:00:00


In [ ]:
%pip check

google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 3.1.1.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
google-cloud-bigquery 3.47.0 has requirement python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0.
holidays 0.108 has requirement python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0.
betterproto 2.0.0b7 has requirement python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0.
dask-cudf-cu13 26.6.0 has requirement pandas<2.4.0,>=2.0, but you have pandas 3.1.1.
gcsfs 2025.12.0 has requirement fsspec==2025.12.0, but you have fsspec 2026.2.0.
gradio-client 2.7.2 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.
bigframes 2.51.0 has requirement python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0.
ibis-framework 9.5.0 has requirement python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0.
gradio 6.31.0 has requirement huggingface-hub<3.0,>=1.16.0, but 

In [ ]:
import huggingface_hub
import diffusers
import transformers
import datasets

print("Hub:", huggingface_hub.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Hub: 0.36.2
Diffusers: 0.35.2
Transformers: 4.57.6
Datasets: 4.8.5


In [ ]:
import torch, os
print('PyTorch:', torch.__version__)
print('GPU available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not available; change the runtime before starting both model trainings.')


PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


##**Step 1 — Construct an annotated NLP dataset**

Every text is paired with an intent and character-offset entity spans. We use 12 intent classes and 23 entity labels, including grafting, quantities, quality, delivery and returns.

The generator separates entire sentence-template families across train/validation/synthetic-test, which is stronger than randomly splitting near-identical generated examples. We still need a genuinely real-world held-out test set.

Our goal is to teach CamemBERT how to recognize linguistic contexts, not to memorize a tiny list of values.
- HUMIDITY: generated over 0–100%, including integer values, decimals and boundaries.
- TEMPERATURE: sampled over 10–40°C for this pilot; change this only after validating local operational limits.
- CITY: 82 illustrative Tunisian city/governorate names, with separate place pools for training, validation and synthetic testing. No city list is exhaustive.
- QUANTITY: varied plant counts (1–50,000), including common round quantities.
- DATE: actual valid calendar dates in multiple formats, with examples crossing campaign boundaries.
- LOT: multiple 5-digit formats such as RO-12345/25 and GRO-G12345/.
- ENTITY LABELS: 24, now including CITY (optional delivery information).
The dataset has 12 intents and 8 phrase families per intent. Entire phrase families, not only records, are held out between train/validation/test. Human-authored, real-world evaluation is still essential. These values are examples for extraction, not agronomic recommendations or business approval rules.

In [ ]:
"""Synthetic, annotated French agricultural commands.

NO company files loaded, NO real customers/employee data. Synthetic data is a
training bootstrap only, NOT a valid production test set.

The format is compatible with SmartTrace's existing backend/nlp/train.py.
"""
from __future__ import annotations
import json, random, re
from collections import Counter
from pathlib import Path

INTENTS = [
    'RECEPTION_SEMENCES','SEMIS','GERMINATION','TRANSFERT','PERTES',
    'INTERVENTION','GREFFAGE','CONTROLE_QUALITE','PREPARATION',
    'LIVRAISON','RETOUR','HORS_SUJET'
]
# Each intent has 8 phrase families, with entire families held out. We hold out entire phrase
# families for validation and synthetic testing, not only individual rows.
TEMPLATES = {
 'RECEPTION_SEMENCES': [
  "Réception de <QUANTITY> semences de <VARIETY> fournies par <SUPPLIER> le <DATE>.",
  "Nous avons reçu <QUANTITY> graines de <VARIETY> du fournisseur <SUPPLIER>.",
  "Enregistrer l'arrivage de <QUANTITY> semences <VARIETY> provenant de <SUPPLIER>.",
  "Le <DATE>, le fournisseur <SUPPLIER> a livré <QUANTITY> graines de <VARIETY>.",
  "Saisir une réception : variété <VARIETY>, quantité <QUANTITY>, fournisseur <SUPPLIER>."
 ],
 'SEMIS': [
  "Semer <QUANTITY> graines de <VARIETY> issues du lot <LOT> en <TRAY> avec <SUBSTRATE>.",
  "Nous avons effectué le semis du lot <LOT> : <QUANTITY> graines, <TRAY>.",
  "Lancer le semis de <QUANTITY> semences du lot <LOT> le <DATE>.",
  "Le <DATE>, mise en plateau de <QUANTITY> graines <VARIETY> du lot <LOT>.",
  "Enregistrer <QUANTITY> graines semées, lot <LOT>, support <TRAY>, substrat <SUBSTRATE>."
 ],
 'GERMINATION': [
  "Germination du lot <LOT> : <QUANTITY> plants à <TEMPERATURE> et <HUMIDITY>.",
  "Nous avons observé <QUANTITY> plants levés pour le lot <LOT> après <DURATION>.",
  "Relever la germination de <QUANTITY> plants du lot <LOT> dans la <LOCATION>.",
  "Au <DATE>, <QUANTITY> plants du lot <LOT> ont germé à <TEMPERATURE>.",
  "Rapport de germination : lot <LOT>, <QUANTITY> plants, durée <DURATION>, humidité <HUMIDITY>."
 ],
 'TRANSFERT': [
  "Transférer <QUANTITY> plants du lot <LOT> de la <ORIGIN> vers la <DESTINATION>.",
  "Nous avons déplacé <QUANTITY> plants du lot <LOT> depuis la <ORIGIN> jusqu'à la <DESTINATION>.",
  "Changer l'emplacement du lot <LOT> : <QUANTITY> plants quittent la <ORIGIN> pour la <DESTINATION>.",
  "Le <DATE>, transfert du lot <LOT> de la <ORIGIN> à la <DESTINATION>, soit <QUANTITY> plants.",
  "Déplacer vers la <DESTINATION> les <QUANTITY> plants actuellement dans la <ORIGIN> du lot <LOT>."
 ],
 'PERTES': [
  "Déclarer la perte de <QUANTITY> plants du lot <LOT> dans la <LOCATION> : <REASON>.",
  "Nous avons perdu <QUANTITY> plants du lot <LOT> en <LOCATION>.",
  "Enregistrer <QUANTITY> plants détruits pour le lot <LOT>, motif : <REASON>.",
  "Au <DATE>, constater <QUANTITY> pertes dans le lot <LOT> situé en <LOCATION>.",
  "Le lot <LOT> a subi <QUANTITY> pertes : <REASON>."
 ],
 'INTERVENTION': [
  "Réaliser un traitement avec <PRODUCT> sur le lot <LOT> dans la <LOCATION>.",
  "Intervention d'entretien pour le lot <LOT> le <DATE> en <LOCATION>.",
  "Nous avons appliqué <PRODUCT> sur le lot <LOT> dans la <LOCATION>.",
  "Consigner un traitement du lot <LOT> le <DATE> avec <PRODUCT>.",
  "Observation et intervention sur le lot <LOT> situé dans la <LOCATION>."
 ],
 'GREFFAGE': [
  "Greffer <QUANTITY> plants du lot <LOT> avec le porte-greffe <ROOTSTOCK_LOT> le <DATE>.",
  "Réaliser le greffage de <QUANTITY> greffons <VARIETY>, lot <LOT>, sur le lot <ROOTSTOCK_LOT>.",
  "Nous avons greffé <QUANTITY> plants du lot <LOT> en utilisant le porte-greffe <ROOTSTOCK_LOT>.",
  "Le <DATE>, greffage du lot <LOT> avec <ROOTSTOCK_LOT> : <QUANTITY> plants concernés.",
  "Enregistrer le greffage de <QUANTITY> plants, greffon <LOT> et porte-greffe <ROOTSTOCK_LOT>."
 ],
 'CONTROLE_QUALITE': [
  "Contrôle qualité du lot <LOT> : <QUANTITY> plants inspectés, état <SANITARY>.",
  "Inspecter <QUANTITY> plants du lot <LOT> pour vérifier l'homogénéité <HOMOGENEITY>.",
  "Le <DATE>, inspection sanitaire du lot <LOT> pour <QUANTITY> plants.",
  "Effectuer le contrôle qualité du lot <LOT> : état <SANITARY>, homogénéité <HOMOGENEITY>.",
  "Vérifier si <QUANTITY> plants du lot <LOT> sont commercialisables après inspection."
 ],
 'PREPARATION': [
  "Préparer <QUANTITY> plants du lot <LOT> pour le client <CLIENT>.",
  "Réserver <QUANTITY> plants du lot <LOT> pour la commande de <CLIENT>.",
  "Préparation de commande : <QUANTITY> plants du lot <LOT>, <TRAY>, client <CLIENT>.",
  "Au <DATE>, conditionner <QUANTITY> plants du lot <LOT> en <PALLET> pour <CLIENT>.",
  "Organiser la préparation du lot <LOT> : <QUANTITY> plants destinés au client <CLIENT>."
 ],
 'LIVRAISON': [
  "Livrer <QUANTITY> plants du lot <LOT> au client <CLIENT> avec <TRANSPORTER>.",
  "Expédier <QUANTITY> plants du lot <LOT> : bon <DELIVERY_NOTE>, véhicule <VEHICLE>.",
  "Le <DATE>, livraison de <QUANTITY> plants du lot <LOT> chez <CLIENT>.",
  "Confirmer l'expédition de <QUANTITY> plants du lot <LOT> avec <TRANSPORTER>.",
  "Enregistrer la livraison : lot <LOT>, <QUANTITY> plants, bon <DELIVERY_NOTE>, client <CLIENT>."
 ],
 'RETOUR': [
  "Retour client de <QUANTITY> plants du lot <LOT> : <REASON>.",
  "Le client <CLIENT> retourne <QUANTITY> plants du lot <LOT>.",
  "Enregistrer <QUANTITY> plants retournés pour le lot <LOT> à cause de <REASON>.",
  "Au <DATE>, un retour de <QUANTITY> plants est signalé pour le lot <LOT>.",
  "Réception du retour client : lot <LOT>, <QUANTITY> plants, motif <REASON>."
 ],
 'HORS_SUJET': [
  "Quel temps fera-t-il à <CITY> pour le service <TEAM> ?",
  "Comment retrouver <DOCUMENT> pour le service <TEAM> ?",
  "Peux-tu expliquer <TOPIC> à l'équipe <TEAM> ?",
  "Où trouver <DOCUMENT> concernant <TOPIC> ?",
  "Je souhaite lire <DOCUMENT> pour <TEAM>."
 ]
}

# Additional phrasing beyond the five original sentence families. The purpose is to
# teach language patterns, not to rely on a fixed set of example values.
EXTRA_TEMPLATES = {
 'RECEPTION_SEMENCES': [
  "Le dépôt de <CITY> a reçu <QUANTITY> semences de <VARIETY> via <SUPPLIER>.",
  "Ajouter une entrée de semences : <QUANTITY> unités, <VARIETY>, fournisseur <SUPPLIER>.",
  "Arrivage daté du <DATE> : <SUPPLIER> nous a remis <QUANTITY> graines de <VARIETY>."],
 'SEMIS': [
  "Aujourd'hui on met <QUANTITY> graines en semis, référence <LOT>, sur <TRAY>.",
  "Saisie semis : <DATE> ; <LOT> ; <QUANTITY> graines ; <SUBSTRATE>.",
  "Sur le <TRAY>, semer <QUANTITY> graines de <VARIETY>, lot source <LOT>."],
 'GERMINATION': [
  "Pour le lot <LOT>, taux d'humidité de <HUMIDITY> et température de <TEMPERATURE> : <QUANTITY> plants levés.",
  "À la <LOCATION>, observer <QUANTITY> germinations, lot <LOT>, sur <DURATION>.",
  "Le <DATE>, nous constatons <QUANTITY> plants germés sur <LOT>, avec <HUMIDITY> d'humidité."],
 'TRANSFERT': [
  "Sortir <QUANTITY> plants de la <ORIGIN> et les placer en <DESTINATION>, référence <LOT>.",
  "Mouvement pour <LOT> : <ORIGIN> -> <DESTINATION>, <QUANTITY> plants.",
  "Aujourd'hui on a bougé <QUANTITY> plants de <LOT> vers la <DESTINATION> depuis la <ORIGIN>."],
 'PERTES': [
  "Écart relevé sur <LOT> : <QUANTITY> plants perdus dans la <LOCATION>.",
  "Nous avons écarté <QUANTITY> plants non viables du lot <LOT> ; cause : <REASON>.",
  "Le <DATE>, déclarer <QUANTITY> pertes de <LOT> dans la <LOCATION>."],
 'INTERVENTION': [
  "Pour <LOT>, consigner une intervention dans la <LOCATION> : produit <PRODUCT>.",
  "Un entretien a été réalisé le <DATE> dans la <LOCATION> pour le lot <LOT>.",
  "Note de traitement pour <LOT> : application de <PRODUCT> en <LOCATION>."],
 'GREFFAGE': [
  "À la date du <DATE>, greffer <QUANTITY> plants du greffon <LOT> sur <ROOTSTOCK_LOT>.",
  "Utiliser <ROOTSTOCK_LOT> comme porte-greffe du lot <LOT> : <QUANTITY> greffages.",
  "Opération de greffe : <LOT> / <ROOTSTOCK_LOT>, nombre de plants <QUANTITY>."],
 'CONTROLE_QUALITE': [
  "Inspecter le lot <LOT> : état sanitaire <SANITARY>, quantité contrôlée <QUANTITY>.",
  "Compte rendu qualité du <DATE> : lot <LOT>, homogénéité <HOMOGENEITY>.",
  "Faire vérifier <QUANTITY> plants sur <LOT> avant toute mise en vente."],
 'PREPARATION': [
  "Pour <CLIENT> à <CITY>, mettre de côté <QUANTITY> plants du lot <LOT>.",
  "Commande à préparer : client <CLIENT>, référence <LOT>, quantité <QUANTITY>.",
  "Conditionner <QUANTITY> plants de <LOT>, en <PALLET>, à destination de <CLIENT>."],
 'LIVRAISON': [
  "Expédier <QUANTITY> plants du lot <LOT> à <CITY> pour <CLIENT>, via <TRANSPORTER>.",
  "Bon <DELIVERY_NOTE> : livraison de <QUANTITY> plants <LOT>, ville <CITY>.",
  "Le <DATE>, le camion <VEHICLE> livre <QUANTITY> plants du lot <LOT> à <CLIENT>."],
 'RETOUR': [
  "Le client <CLIENT> signale un retour de <QUANTITY> plants pour <LOT>.",
  "Réintégrer en quarantaine <QUANTITY> plants retournés du lot <LOT> : <REASON>.",
  "Réclamation du <DATE> concernant <LOT> : <QUANTITY> plants revenus."],
 'HORS_SUJET': [
  "Ne pas transférer les <QUANTITY> plants du lot <LOT> de la <ORIGIN> vers la <DESTINATION>.",
  "Peux-tu me dire pourquoi les plants du lot <LOT> ont été perdus ?",
  "Faut-il expédier <QUANTITY> plants vers <CITY> aujourd'hui ?"]
}
for intent, more in EXTRA_TEMPLATES.items():
    TEMPLATES[intent].extend(more)

# Negated instructions/questions should NOT be misclassified as executable actions.
TEMPLATES['HORS_SUJET'][1] = "Ne lancez pas le semis de <QUANTITY> graines du lot <LOT>."
TEMPLATES['HORS_SUJET'][2] = "Est-ce qu'on doit livrer <QUANTITY> plants du lot <LOT> à <CITY> ?"
# Ensure CITY is learned within actionable intents, not only weather/FAQ text.
TEMPLATES['PREPARATION'][0] = "Préparer <QUANTITY> plants du lot <LOT> pour le client <CLIENT> à <CITY>."
TEMPLATES['LIVRAISON'][0] = "Livrer <QUANTITY> plants du lot <LOT> au client <CLIENT> à <CITY> avec <TRANSPORTER>."

PATTERN = re.compile(r'<([A-Z_]+)>')

# Broad (not exhaustive) Tunisian geographical vocabulary: governorates and towns.
# Never hardcode these as the only valid cities at inference time.
CITIES = [
    'Tunis','Ariana','Ben Arous','La Manouba','Nabeul','Zaghouan',
    'Bizerte','Béja','Jendouba','Le Kef','Siliana','Sousse',
    'Monastir','Mahdia','Sfax','Kairouan','Kasserine','Sidi Bouzid',
    'Gabès','Médenine','Tataouine','Gafsa','Tozeur','Kébili',
    'Hammamet','Korba','Kélibia','Menzel Temime','Soliman','Grombalia',
    'Dar Chaabane','La Marsa','Le Bardo','Carthage','Raoued','Mnihla',
    'Ettadhamen','Ezzahra','Hammam Lif','Radès','Mornag','Mégrine',
    'Menzel Bourguiba','Ras Jebel','Mateur','Tabarka','Aïn Draham',
    'Hammam Sousse','Akouda','Msaken','Enfidha','Kalaa Kebira',
    'Kalaa Seghira','Ksar Hellal','Moknine','Jemmal','Sahline',
    'El Jem','Ksour Essef','Chebba','Sakiet Ezzit','Sakiet Eddaier',
    'Mahres','Jebiniana','Kerkennah','Hajeb El Ayoun','Haffouz',
    'Fériana','Sbeitla','Regueb','Meknassy','Métlaoui','Redeyef',
    'El Hamma','Mareth','Djerba Houmt Souk','Midoun','Zarzis',
    'Ben Guerdane','Douz','Nefta','Degache'
]
# Hold out a portion of place names, in addition to whole sentence families.
CITY_POOLS = {'train': CITIES[:62], 'validation': CITIES[62:72],
              'test_synthetic': CITIES[72:]}

VALUES = {
    'TEAM':['logistique','administration','informatique','formation','achats','support','production','qualité'],
    'DOCUMENT':['la documentation du logiciel','le guide de connexion','le rapport annuel',
                'le manuel utilisateur','les consignes internes','le plan de formation',
                'le guide des lots','les paramètres du système','la note de service', 'le document de qualité'],
    'TOPIC':['les paramètres du compte','le tableau de bord','les notifications',
             'les rôles utilisateurs','la page d’accueil','le fonctionnement de la production',
             'les accès au site','la langue de l’application','la gestion des alertes','le menu principal'],
    'VARIETY':['tomate F1','aubergine F1','poivron F1','pastèque F1','courgette F1',
               'tomate Roma','tomate cerise','tomate ronde','poivron doux','piment fort',
               'melon charentais','concombre F1','courge F1','laitue','brocoli F1',
               'chou-fleur F1','tomate A336 F1','aubergine Black Beauty','pastèque Crimson',
               'tomate hybride T12','poivron hybride P8'],
    'SUBSTRATE':['substrat coco','substrat tourbe','substrat horticole','fibres de coco',
                 'terreau de semis','mélange tourbe-perlite'],
    'REASON':['croissance irrégulière','manque de vigueur','problème de qualité',
              'feuilles abîmées','développement insuffisant','dessèchement',
              'plants fragiles','anomalie observée','cassure lors du transport'],
    'PRODUCT':['produit agricole A','produit agricole B','solution de traitement C',
               'solution témoin D','produit de démonstration E'],
    'SANITARY':['satisfaisant','à contrôler','non conforme','sain','à recontrôler','douteux'],
    'HOMOGENEITY':['homogène','hétérogène','variable','régulière','irrégulière'],
}


def _fmt_number(value: float, rng: random.Random, *, decimals_allowed=True) -> str:
    if decimals_allowed and rng.random() < 0.38:
        s = f'{value:.1f}'
        return s.replace('.', ',') if rng.random() < 0.85 else s
    return str(int(round(value)))


def _lot_reference(rng: random.Random) -> str:
    code = rng.randint(10000, 99999)
    yy = rng.choice([23, 24, 25, 26, 27])
    return rng.choice([f'RO-{code}/{yy:02d}', f'GRO-G{code}/', f'{code}', f'RO-{code:05d}/{yy:02d}'])


def value_for(label: str, rng: random.Random, *, split='train', context=None) -> str:
    """Sample varied, format-valid entity values; not agricultural recommendations."""
    context = context if context is not None else {}
    if label == 'CITY':
        pool = CITY_POOLS[split]
        return rng.choice(pool)
    if label in ('LOT', 'ROOTSTOCK_LOT'):
        if label not in context:
            other = 'LOT' if label == 'ROOTSTOCK_LOT' else 'ROOTSTOCK_LOT'
            value = _lot_reference(rng)
            while value == context.get(other):
                value = _lot_reference(rng)
            context[label] = value
        return context[label]
    if label == 'QUANTITY':
        if rng.random() < 0.15:
            return str(rng.choice([1, 2, 10, 40, 104, 500, 1000, 5000, 10000, 25000]))
        return str(rng.randint(1, 50000))
    if label == 'TEMPERATURE':
        # User-requested example range: 10...40 °C inclusive.
        if rng.random() < 0.15:
            v = rng.choice([10, 11, 15, 20, 25, 30, 35, 39, 40])
        elif rng.random() < 0.4:
            v = rng.randint(10, 40)
        else:
            v = rng.randint(100, 400) / 10
        return f'{_fmt_number(v, rng)} {rng.choice(["°C", "degrés", "C"])}'
    if label == 'HUMIDITY':
        # User-requested mathematical range: 0...100 % inclusive.
        if rng.random() < 0.17:
            v = rng.choice([0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100])
        elif rng.random() < 0.40:
            v = rng.randint(0, 100)
        else:
            v = rng.randint(0, 1000) / 10
        return f'{_fmt_number(v, rng)} {rng.choice(["%", "pour cent"])}'
    if label == 'DURATION':
        return f'{rng.randint(2, 240)} {rng.choice(["heures", "h"])}'
    if label == 'DATE':
        from datetime import date, timedelta
        start = date(2023, 1, 1)
        end = date(2027, 12, 31)
        d = start + timedelta(days=rng.randint(0, (end-start).days))
        return d.strftime('%d/%m/%Y') if rng.random() < 0.65 else d.isoformat()
    if label == 'TRAY':
        return f'{rng.choice(["plateau", "plaque", "alvéoles P"])} {rng.choice([40, 60, 72, 84, 96, 104, 128, 150, 200, 240, 288])}'
    if label == 'PALLET':
        n = rng.randint(1, 50)
        return f'{n} {"palette" if n == 1 else "palettes"}'
    if label in ('ORIGIN', 'DESTINATION', 'LOCATION'):
        if label == 'LOCATION':
            return f'{rng.choice(["chambre", "serre", "zone", "tunnel"])} {rng.randint(1, 40)}'
        if 'location_pair' not in context:
            left = f'{rng.choice(["chambre", "serre", "zone", "tunnel"])} {rng.randint(1, 40)}'
            right = f'{rng.choice(["chambre", "serre", "zone", "tunnel"])} {rng.randint(1, 40)}'
            while right == left:
                right = f'{rng.choice(["chambre", "serre", "zone", "tunnel"])} {rng.randint(1, 40)}'
            context['location_pair'] = {'ORIGIN': left, 'DESTINATION': right}
        return context['location_pair'][label]
    if label == 'SUPPLIER':
        return f'Fournisseur Démo {rng.randint(1, 900)}'
    if label == 'CLIENT':
        return f'Exploitation Démo {rng.randint(1, 2000)}'
    if label == 'TRANSPORTER':
        return f'Transporteur Démo {rng.randint(1, 800)}'
    if label == 'VEHICLE':
        return f'{rng.choice(["véhicule", "camion", "utilitaire"])} DEMO-{rng.randint(1, 999):03d}'
    if label == 'DELIVERY_NOTE':
        return f'BL-DEMO-{rng.randint(100, 99999)}'
    if label in VALUES:
        return rng.choice(VALUES[label])
    raise KeyError(f'Unknown placeholder: {label}')


def render(template: str, rng: random.Random, *, split='train', overrides=None):
    chunks, entities = [], []
    cursor, total = 0, 0
    context = {}
    overrides = overrides or {}
    for match in PATTERN.finditer(template):
        plain = template[cursor:match.start()]
        chunks.append(plain)
        total += len(plain)
        label = match.group(1)
        value = overrides.get(label) or value_for(label, rng, split=split, context=context)
        if label not in {'TEAM', 'DOCUMENT', 'TOPIC'}:
            entities.append({'start':total,'end':total+len(value), 'label':label})
        chunks.append(value)
        total += len(value)
        cursor=match.end()
    chunks.append(template[cursor:])
    text=''.join(chunks)
    for ent in entities:
        assert text[ent['start']:ent['end']]
    return text, entities


def build_dataset(output_dir='data/nlp', samples_train=40, samples_eval=25, seed=43):
    """Synthetic bootstrap. Split by disjoint phrase families and city vocabularies."""
    rng = random.Random(seed)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    partitions = {s:[] for s in ('train', 'validation', 'test_synthetic')}
    seen_all = set()
    for intent, templates in TEMPLATES.items():
        assert len(templates) == 8, intent
        for index, template in enumerate(templates):
            split = 'train' if index < 5 else ('validation' if index == 5 else 'test_synthetic')
            target_n = samples_train if split == 'train' else samples_eval
            n_inserted = 0
            for _ in range(target_n * 20):
                if n_inserted >= target_n:
                    break
                text, ents = render(template, rng, split=split)
                if text in seen_all:
                    continue
                seen_all.add(text)
                partitions[split].append({'text':text,'intent':intent,'entities':ents,
                     'source':'synthetic','template_family':f'{intent}:{index}'})
                n_inserted += 1
            if n_inserted != target_n:
                raise RuntimeError(f'Not enough unique samples for {intent} template {index}: {n_inserted}/{target_n}')
    # Explicit boundary cases ensure that endpoints are represented, not only
    # theoretically reachable by a random draw.
    for label, forced in [('HUMIDITY','0 %'),('HUMIDITY','100 %'),
                           ('TEMPERATURE','10 °C'),('TEMPERATURE','40 °C')]:
        text, ents = render(TEMPLATES['GERMINATION'][0], rng, split='train',
                            overrides={label:forced})
        assert text not in seen_all
        seen_all.add(text)
        partitions['train'].append({'text':text,'intent':'GERMINATION',
                                    'entities':ents,'source':'synthetic',
                                    'template_family':'GERMINATION:0',
                                    'scenario':'boundary_'+label.lower()})
    stats = {}
    for split, records in partitions.items():
        rng.shuffle(records)
        path=out/f'{split}.jsonl'
        with path.open('w', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record,ensure_ascii=False)+'\n')
        stats[split] = {'path':str(path),'rows':len(records),
                        'intents':dict(Counter(r['intent'] for r in records))}
    for x in partitions:
        for y in partitions:
            if x >= y: continue
            assert not {r['text'] for r in partitions[x]} & {r['text'] for r in partitions[y]}
            assert not {r['template_family'] for r in partitions[x]} & {r['template_family'] for r in partitions[y]}
    card={'status':'synthetic_bootstrap_not_real_validation',
          'split_policy':'disjoint phrase families; disjoint city vocabularies',
          'sampling':{'humidity_pct':[0,100],'temperature_c':[10,40],
                      'quantity_plants':[1,50000], 'duration_hours':[2,240],
                      'geographical_scope':'illustrative Tunisian cities and governorates, not exhaustive'},
          'note':'Ranges train extraction. They do not define acceptable crop conditions or validate business events.',
          'result':stats}
    (out/'dataset_card.json').write_text(json.dumps(card,ensure_ascii=False,indent=2),encoding='utf-8')
    return stats

if __name__ == '__main__':
    print(json.dumps(build_dataset(),ensure_ascii=False,indent=2))

{
  "train": {
    "path": "data/nlp/train.jsonl",
    "rows": 2404,
    "intents": {
      "CONTROLE_QUALITE": 200,
      "RETOUR": 200,
      "RECEPTION_SEMENCES": 200,
      "SEMIS": 200,
      "GERMINATION": 204,
      "GREFFAGE": 200,
      "LIVRAISON": 200,
      "PREPARATION": 200,
      "TRANSFERT": 200,
      "INTERVENTION": 200,
      "PERTES": 200,
      "HORS_SUJET": 200
    }
  },
  "validation": {
    "path": "data/nlp/validation.jsonl",
    "rows": 300,
    "intents": {
      "PREPARATION": 25,
      "TRANSFERT": 25,
      "GREFFAGE": 25,
      "PERTES": 25,
      "CONTROLE_QUALITE": 25,
      "GERMINATION": 25,
      "RECEPTION_SEMENCES": 25,
      "RETOUR": 25,
      "SEMIS": 25,
      "LIVRAISON": 25,
      "HORS_SUJET": 25,
      "INTERVENTION": 25
    }
  },
  "test_synthetic": {
    "path": "data/nlp/test_synthetic.jsonl",
    "rows": 600,
    "intents": {
      "TRANSFERT": 50,
      "INTERVENTION": 50,
      "GREFFAGE": 50,
      "LIVRAISON": 50,
      "CONTROLE_

In [ ]:
from collections import Counter
splits=build_dataset('data/nlp',samples_train=40,samples_eval=25,seed=43)
for split,v in splits.items():
    print(f"{split}: {v['rows']} sentences, {len(v['intents'])} intents")
print('Note: four extra training examples explicitly include humidity and temperature boundaries.')

train: 2404 sentences, 12 intents
validation: 300 sentences, 12 intents
test_synthetic: 600 sentences, 12 intents
Note: four extra training examples explicitly include humidity and temperature boundaries.


In [ ]:
# Inspect how much the values vary before training. This needs no GPU.
import re, json
from pathlib import Path
records = [json.loads(line) for part in ('train','validation','test_synthetic')
           for line in Path(f'data/nlp/{part}.jsonl').read_text(encoding='utf-8').splitlines()]
for label in ('HUMIDITY','TEMPERATURE','CITY','QUANTITY','DATE'):
    values=[r['text'][e['start']:e['end']] for r in records for e in r['entities'] if e['label']==label]
    print(label, ':', len(values),'occurrences /',len(set(values)),'distinct values; examples:',values[:5])
    if label in ('HUMIDITY','TEMPERATURE'):
        nums=[float(re.search(r'\d+(?:[.,]\d+)?',v).group().replace(',','.')) for v in values]
        lo,hi=(0,100) if label=='HUMIDITY' else (10,40)
        assert min(nums)>=lo and max(nums)<=hi
        assert min(nums)==lo and max(nums)==hi
        print('  observed numeric range:',min(nums),'to',max(nums))
print('All 12 intent classes and all 24 entity labels are checked when train_one() loads data.')

HUMIDITY : 134 occurrences / 103 distinct values; examples: ['36 pour cent', '71 pour cent', '46 %', '46 %', '99 pour cent']
  observed numeric range: 0.0 to 100.0
TEMPERATURE : 109 occurrences / 82 distinct values; examples: ['24 °C', '35 degrés', '36.2 C', '11,0 C', '31 degrés']
  observed numeric range: 10.0 to 40.0
CITY : 285 occurrences / 79 distinct values; examples: ['Sakiet Eddaier', 'Carthage', 'Sahline', 'Ras Jebel', 'Sakiet Ezzit']
QUANTITY : 2819 occurrences / 2373 distinct values; examples: ['40', '30312', '37694', '34656', '5815']
DATE : 825 occurrences / 723 distinct values; examples: ['11/05/2023', '2024-03-04', '05/10/2025', '2023-12-23', '25/12/2024']
All 12 intent classes and all 24 entity labels are checked when train_one() loads data.


In [ ]:
import json
from pathlib import Path
for line in Path('data/nlp/train.jsonl').read_text(encoding='utf-8').splitlines()[:4]:
    r=json.loads(line)
    print(r['intent'],':',r['text'])
    print('entities:',[(r['text'][e['start']:e['end']],e['label']) for e in r['entities']])
    print()

CONTROLE_QUALITE : Inspecter 40 plants du lot 63941 pour vérifier l'homogénéité homogène.
entities: [('40', 'QUANTITY'), ('63941', 'LOT'), ('homogène', 'HOMOGENEITY')]

RETOUR : Réception du retour client : lot 77801, 30312 plants, motif plants fragiles.
entities: [('77801', 'LOT'), ('30312', 'QUANTITY'), ('plants fragiles', 'REASON')]

RECEPTION_SEMENCES : Réception de 37694 semences de tomate Roma fournies par Fournisseur Démo 386 le 11/05/2023.
entities: [('37694', 'QUANTITY'), ('tomate Roma', 'VARIETY'), ('Fournisseur Démo 386', 'SUPPLIER'), ('11/05/2023', 'DATE')]

SEMIS : Enregistrer 34656 graines semées, lot RO-55571/25, support plateau 40, substrat fibres de coco.
entities: [('34656', 'QUANTITY'), ('RO-55571/25', 'LOT'), ('plateau 40', 'TRAY'), ('fibres de coco', 'SUBSTRATE')]



##**Step 2 — Train two separate CamemBERT heads**

**Intent model:** CamemBERT encoder + sequence-classification head (12 classes). Optimize cross-entropy; measure accuracy + macro-F1.

**NER model:** CamemBERT encoder + token-classification head with BIO labels. Align source character offsets with fast-tokenizer subwords; ignore special-token labels using -100. Evaluate entity boundaries using seqeval precision/recall/F1.

The initial learning rate is 2e-5, batch size 8, three epochs, and best checkpoint selected on validation F1. These are experiment defaults, not tuned guarantees

In [ ]:
"""CamemBERT sequence-classification + BIO token-classification trainer.

Designed to run in Google Colab with GPU. Run notebook rather than this file
for a documented interactive training experience.

Company data must NOT be sent to Colab without organizational authorization.
"""
from __future__ import annotations
import json
from pathlib import Path
from collections import Counter

INTENTS = ['HORS_SUJET','RECEPTION_SEMENCES','SEMIS','GERMINATION','TRANSFERT',
           'PERTES','INTERVENTION','GREFFAGE','CONTROLE_QUALITE','PREPARATION',
           'LIVRAISON','RETOUR']
ENTITIES = ['LOT','QUANTITY','ORIGIN','DESTINATION','LOCATION','TEMPERATURE',
            'HUMIDITY','DURATION','PRODUCT','CLIENT','DATE','ROOTSTOCK_LOT',
            'VARIETY','SUPPLIER','TRAY','SUBSTRATE','REASON','SANITARY',
            'HOMOGENEITY','PALLET','TRANSPORTER','VEHICLE','DELIVERY_NOTE','CITY']


def load_rows(path):
    rows=[json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]
    assert rows, f'Empty dataset {path}'
    for row in rows:
        assert row['intent'] in INTENTS
        assert row['text']
        prev=0
        for s in sorted(row['entities'],key=lambda x:x['start']):
            assert s['label'] in ENTITIES
            assert 0<=s['start']<s['end']<=len(row['text'])
            assert s['start']>=prev, ('Overlapping annotation',row)
            prev=s['end']
    return rows


def load_splits(data_dir='data/nlp'):
    p=Path(data_dir)
    spl={name:load_rows(p/f'{name}.jsonl') for name in ('train','validation','test_synthetic')}
    for a,b in [('train','validation'),('train','test_synthetic'),('validation','test_synthetic')]:
        assert not ({row['text'] for row in spl[a]} & {row['text'] for row in spl[b]}), f'Text leakage {a}/{b}'
        assert not ({row['template_family'] for row in spl[a]} & {row['template_family'] for row in spl[b]}), f'Template leakage {a}/{b}'
    return spl


def train_one(task: str, *, data_dir='data/nlp', model_name='camembert-base', epochs=3,
              base_dir='models', batch_size=8):
    assert task in ('intent','entities')
    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    from seqeval.metrics import f1_score as ner_f1, precision_score as ner_precision, recall_score as ner_recall
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,AutoModelForTokenClassification,
        Trainer,TrainingArguments,DataCollatorWithPadding,DataCollatorForTokenClassification,
    )
    spl=load_splits(data_dir)
    labels=INTENTS if task=='intent' else ['O']+[f'{x}-{e}' for e in ENTITIES for x in ('B','I')]
    l2i={name:i for i,name in enumerate(labels)}
    tokenizer=AutoTokenizer.from_pretrained(model_name,use_fast=True)
    assert tokenizer.is_fast, 'Fast tokenizer required to align BIO spans'

    def encode_one(row):
        encoding=tokenizer(row['text'], truncation=True, max_length=256,
                           return_offsets_mapping=(task=='entities'))
        if task=='intent':
            encoding['labels']=l2i[row['intent']]
        else:
            offsets=encoding.pop('offset_mapping')
            tags=[];previous=None
            ordered=sorted(row['entities'],key=lambda x:x['start'])
            for a,b in offsets:
                if a==b:tags.append(-100);previous=None;continue
                overlapping=[e for e in ordered if a<e['end'] and b>e['start']]
                if not overlapping:
                    tags.append(l2i['O']);previous=None;continue
                if len(overlapping)!=1 or not (a>=overlapping[0]['start'] and b<=overlapping[0]['end']):
                    tags.append(-100);previous=None;continue
                entity=overlapping[0]
                sig=(entity['start'],entity['end'],entity['label'])
                prefix='I' if previous==sig else 'B'
                tags.append(l2i[f'{prefix}-{entity["label"]}']);previous=sig
            encoding['labels']=tags
        return encoding

    class Rows(torch.utils.data.Dataset):
        def __init__(self,records):self.examples=[encode_one(x) for x in records]
        def __len__(self):return len(self.examples)
        def __getitem__(self,index):return self.examples[index]

    train_ds=Rows(spl['train']);valid_ds=Rows(spl['validation']);test_ds=Rows(spl['test_synthetic'])
    if task=='intent':
        model=AutoModelForSequenceClassification.from_pretrained(
            model_name,num_labels=len(labels),label2id=l2i,id2label={i:x for i,x in enumerate(labels)})
        collator=DataCollatorWithPadding(tokenizer)
        def compute_metrics(pred):
            yhat=np.argmax(pred.predictions,axis=-1);y=pred.label_ids
            return {'accuracy':float(accuracy_score(y,yhat)),
                    'macro_f1':float(f1_score(y,yhat,labels=list(range(len(labels))),average='macro',zero_division=0))}
    else:
        model=AutoModelForTokenClassification.from_pretrained(
            model_name,num_labels=len(labels),label2id=l2i,id2label={i:x for i,x in enumerate(labels)})
        collator=DataCollatorForTokenClassification(tokenizer)
        def compute_metrics(pred):
            pred_ids=np.argmax(pred.predictions,axis=-1)
            pred_tags=[];true_tags=[]
            for guess,gold in zip(pred_ids,pred.label_ids):
                allowed=[(int(p),int(g)) for p,g in zip(guess,gold) if g!=-100]
                pred_tags.append([labels[p] for p,g in allowed])
                true_tags.append([labels[g] for p,g in allowed])
            return {'entity_f1':float(ner_f1(true_tags,pred_tags,zero_division=0)),
                    'entity_precision':float(ner_precision(true_tags,pred_tags,zero_division=0)),
                    'entity_recall':float(ner_recall(true_tags,pred_tags,zero_division=0))}

    model_out=Path(base_dir)/('nlp-intent' if task=='intent' else 'nlp-entities')
    ckpt_out=Path(base_dir)/('checkpoints-'+task)
    args=TrainingArguments(
        output_dir=str(ckpt_out),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,learning_rate=2e-5,weight_decay=0.01,
        eval_strategy='epoch',save_strategy='epoch',save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_macro_f1' if task=='intent' else 'eval_entity_f1',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),report_to='none',logging_steps=30,
    )
    trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=valid_ds,
                    data_collator=collator,processing_class=tokenizer,
                    compute_metrics=compute_metrics)
    trainer.train()
    validation_metrics=trainer.evaluate(valid_ds,metric_key_prefix='validation')
    synthetic_metrics=trainer.evaluate(test_ds,metric_key_prefix='synthetic_test')
    trainer.save_model(str(model_out))
    tokenizer.save_pretrained(str(model_out))
    result={'task':task,'backbone':model_name,'epochs':epochs,
            'validation':validation_metrics,'synthetic_test':synthetic_metrics,
            'warning':'Synthetic held-out wording is NOT a real human-language generalization benchmark.'}
    if task=='intent':
        pred=trainer.predict(test_ds)
        yhat=np.argmax(pred.predictions,axis=-1)
        result['synthetic_classification_report']=classification_report(
            pred.label_ids,yhat,labels=list(range(len(labels))),target_names=labels,
            zero_division=0,output_dict=True)
    (model_out/'evaluation_report.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    print('Saved',task,'model:',model_out)
    print('Validation:',{k:round(float(v),4) for k,v in validation_metrics.items() if 'f1' in k or 'accuracy' in k})
    print('Synthetic test:',{k:round(float(v),4) for k,v in synthetic_metrics.items() if 'f1' in k or 'accuracy' in k})
    return result

In [ ]:
%pip install seqeval
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
intent_results=train_one('intent',data_dir='data/nlp',model_name='camembert-base',
                         epochs=3,base_dir='models',batch_size=8)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=8db8d89c686ef8bc4b2407bade723c51a1e971264942a7840dc12942adda9fba
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.010800,1.111626,0.916667,0.888889
2,0.428800,0.659731,0.916667,0.888889
3,0.303200,0.581676,0.916667,0.888889


Saved intent model: models/nlp-intent
Validation: {'validation_accuracy': 0.9167, 'validation_macro_f1': 0.8889}
Synthetic test: {'synthetic_test_accuracy': 0.9433, 'synthetic_test_macro_f1': 0.9425}


In [ ]:
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
ner_results=train_one('entities',data_dir='data/nlp',model_name='camembert-base',
                      epochs=3,base_dir='models',batch_size=8)

Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Entity F1,Entity Precision,Entity Recall
1,1.824200,1.848366,0.631141,0.582181,0.689091
2,1.379400,1.421385,0.702479,0.643939,0.772727
3,1.234800,1.302528,0.779331,0.729580,0.836364


Saved entities model: models/nlp-entities
Validation: {'validation_entity_f1': 0.7793}
Synthetic test: {'synthetic_test_entity_f1': 0.7171}


##**Step 3 — Inspect predictions (not proof of reliability)**

A real held-out human-authored dataset must be used before approving operational use. Check missed entities, wrong lots, inconsistent origin/destination roles, incorrect negation interpretation, and ambiguous multi-operation sentences.

In [ ]:
from transformers import pipeline
sentence='Nous avons déplacé 230 plants du lot RO-54321/25 de la chambre 1 vers la serre 5.'
intent_pipe=pipeline('text-classification',model='models/nlp-intent',tokenizer='models/nlp-intent',device=0)
entity_pipe=pipeline('token-classification',model='models/nlp-entities',tokenizer='models/nlp-entities',aggregation_strategy='simple',device=0)
print('Sentence:',sentence)
print('Intent:',intent_pipe(sentence))
print('Entities:',entity_pipe(sentence))

Device set to use cuda:0
Device set to use cuda:0


Sentence: Nous avons déplacé 230 plants du lot RO-54321/25 de la chambre 1 vers la serre 5.
Intent: [{'label': 'TRANSFERT', 'score': 0.4204113483428955}]
Entities: [{'entity_group': 'QUANTITY', 'score': np.float32(0.3064682), 'word': '230', 'start': 19, 'end': 22}, {'entity_group': 'LOT', 'score': np.float32(0.34829995), 'word': 'RO-54321/25', 'start': 37, 'end': 48}, {'entity_group': 'LOCATION', 'score': np.float32(0.07630243), 'word': 'chambre', 'start': 55, 'end': 62}, {'entity_group': 'ORIGIN', 'score': np.float32(0.070836365), 'word': '1', 'start': 63, 'end': 64}, {'entity_group': 'DESTINATION', 'score': np.float32(0.07799428), 'word': 'serre 5', 'start': 73, 'end': 80}]


In [ ]:
test_sentences = [
    "Nous avons déplacé 230 plants du lot RO-54321/25 de la chambre 1 vers la serre 5.",
    "Transférer 150 tomates de la serre 2 vers la chambre 3.",
    "Le lot RO-12345/24 a été déplacé de la zone A vers la zone B.",
    "Nous avons transféré 80 plants vers la serre 4.",
    "Déplacer 300 plants de la chambre froide vers la serre principale."
]

for text in test_sentences:
    print("\nSentence:", text)
    print("Intent:", intent_pipe(text, top_k=None))
    print("Entities:", entity_pipe(text))


Sentence: Nous avons déplacé 230 plants du lot RO-54321/25 de la chambre 1 vers la serre 5.
Intent: [{'label': 'TRANSFERT', 'score': 0.4204113483428955}, {'label': 'INTERVENTION', 'score': 0.06548342108726501}, {'label': 'SEMIS', 'score': 0.06531377136707306}, {'label': 'PERTES', 'score': 0.0649544820189476}, {'label': 'RETOUR', 'score': 0.06042709946632385}, {'label': 'HORS_SUJET', 'score': 0.05353080853819847}, {'label': 'LIVRAISON', 'score': 0.04890210181474686}, {'label': 'PREPARATION', 'score': 0.04767619073390961}, {'label': 'GERMINATION', 'score': 0.04459017515182495}, {'label': 'CONTROLE_QUALITE', 'score': 0.044511280953884125}, {'label': 'RECEPTION_SEMENCES', 'score': 0.043219100683927536}, {'label': 'GREFFAGE', 'score': 0.04098028317093849}]
Entities: [{'entity_group': 'QUANTITY', 'score': np.float32(0.3064682), 'word': '230', 'start': 19, 'end': 22}, {'entity_group': 'LOT', 'score': np.float32(0.34829995), 'word': 'RO-54321/25', 'start': 37, 'end': 48}, {'entity_group': 'LO

##**Step 4 — Export to SmartTrace**

In [ ]:
import zipfile
from pathlib import Path
output_file=Path('smarttrace_camembert_models.zip')
with zipfile.ZipFile(output_file,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for directory in [Path('models/nlp-intent'),Path('models/nlp-entities')]:
        for file in directory.rglob('*'):
            if file.is_file():archive.write(file,file.as_posix())
    archive.write('data/nlp/dataset_card.json','dataset_card.json')
print('Export ready:',output_file,'size (MB):',round(output_file.stat().st_size/1024**2,1))

Export ready: smarttrace_camembert_models.zip size (MB): 737.1


In [ ]:
from google.colab import files
# The trained model ZIP may be large. Alternatively save it to Google Drive.
files.download('smarttrace_camembert_models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>